# Day 19 — Confidence Intervals
> Quantifying uncertainty around estimates.

## What is a Confidence Interval?

A 95% CI means: if we repeated this experiment 100 times, approximately 95 of the resulting intervals would contain the true population parameter.

> It does **not** mean there is a 95% probability the true value is in this specific interval.

## Formula (for means)
$$CI = \bar{x} \pm t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}}$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style='whitegrid')
np.random.seed(42)

# Generate sample data: daily revenue
revenue = np.random.normal(loc=127.5, scale=30, size=80)
n = len(revenue)
mean = revenue.mean()
se = stats.sem(revenue)
ci95 = stats.t.interval(0.95, df=n-1, loc=mean, scale=se)
ci99 = stats.t.interval(0.99, df=n-1, loc=mean, scale=se)

print(f"Sample mean      : ${mean:.2f}")
print(f"Standard error   : ${se:.2f}")
print(f"95% CI           : (${ci95[0]:.2f}, ${ci95[1]:.2f})  width={ci95[1]-ci95[0]:.2f}")
print(f"99% CI           : (${ci99[0]:.2f}, ${ci99[1]:.2f})  width={ci99[1]-ci99[0]:.2f}")


In [ ]:
# Visualize: CI width vs confidence level
levels = np.arange(0.70, 0.9999, 0.01)
widths = []
for level in levels:
    ci = stats.t.interval(level, df=n-1, loc=mean, scale=se)
    widths.append(ci[1] - ci[0])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Width vs confidence level
axes[0].plot(levels * 100, widths, color='steelblue', lw=2)
axes[0].axvline(95, color='red', linestyle='--', lw=1.5, label='95%')
axes[0].axvline(99, color='orange', linestyle='--', lw=1.5, label='99%')
axes[0].set_xlabel('Confidence Level (%)')
axes[0].set_ylabel('CI Width ($)')
axes[0].set_title('Wider confidence = More certainty')
axes[0].legend()

# Simulation: 50 CIs — how many capture true mean?
true_mean = 127.5
captured = 0
for i in range(50):
    sample = np.random.normal(true_mean, 30, 80)
    ci = stats.t.interval(0.95, df=79, loc=sample.mean(), scale=stats.sem(sample))
    color = '#2ecc71' if ci[0] <= true_mean <= ci[1] else '#e74c3c'
    axes[1].plot([ci[0], ci[1]], [i, i], color=color, lw=1.5, alpha=0.7)
    if ci[0] <= true_mean <= ci[1]: captured += 1

axes[1].axvline(true_mean, color='black', lw=2, label=f'True mean={true_mean}')
axes[1].set_title(f'50 CIs: {captured}/50 captured true mean ({captured}%)')
axes[1].legend()
axes[1].set_xlabel('Revenue ($)')

plt.suptitle('Confidence Intervals Visualization', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/05_confidence_interval.png', dpi=150)
plt.show()


In [ ]:
# CIs for proportion (binomial)
conversions = 43
n_users = 500
p_hat = conversions / n_users
se_prop = np.sqrt(p_hat * (1 - p_hat) / n_users)
z = 1.96  # for 95%
ci_prop = (p_hat - z * se_prop, p_hat + z * se_prop)

print(f"Conversion rate: {p_hat:.3%}")
print(f"95% CI: ({ci_prop[0]:.3%}, {ci_prop[1]:.3%})")
